# Agent Learning Kit — SDK Demo **v2** (one spec, one runner)

`v1` (`agent_learning_sdk_demo.ipynb`) toured the SDK through the manifest helpers.
This **v2** shows the SDK the way it is actually built underneath — an
**environment / actor / runner** harness where every piece is plug-and-play:

| Piece | In this SDK | You supply |
|---|---|---|
| **Environment** | a registered `EnvironmentPlugin` (`chat`, `voice`, or your own) that owns the world + action space | pick one, or `@register_environment` your own |
| **Actor / agent** | an **ActorSource** (`system_prompt` \| callable \| `factory` \| `http` \| `framework`) resolved through one registry — *or* any object with `.call()` | **drop in ANY agent** |
| **Episode state** | a `Scenario` (personas + situations + desired outcomes) | describe the test |
| **Contract** | a frozen `SimulationSpec` tying environment + target + simulator together | declarative, secret-free |
| **Runner** | **one** `SimulationRunner` — the *same* spine for chat **and** voice | `.run(spec, target=..., environment=...)` |

Everything below imports from the `fi.alk.simulate` facade (aliased `S`). Offline
cells always run. Live cells (real LLM, real voice calls, platform submit) gate on
credentials and an opt-in flag.

## Setup

In [ ]:
import asyncio, json, os
from pathlib import Path


def load_env_file(path):
    # Load KEY=VALUE lines from an env file into os.environ (values stay local).
    p = Path(path).expanduser()
    if not p.exists():
        return False
    for line in p.read_text().splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        k, v = line.split("=", 1)
        os.environ.setdefault(k.strip(), v.strip().strip('"'))
    return True

# IMPORTANT: load creds BEFORE importing the SDK. `fi.alk.config` binds FI_BASE_URL
# at import time — if the SDK imports first while FI_BASE_URL is unset, it pins the
# public api.futureagi.com default and later fi_eval calls ignore your platform.
# The acceptance env carries Vertex + LiveKit/Deepgram + Vapi/Retell + FI creds.
loaded = load_env_file(os.environ.get("ACCEPTANCE_ENV_FILE", "../.env.acceptance"))

import fi.alk.simulate as S
from fi.simulate.agent.wrapper import AgentInput, AgentResponse

VERTEX_READY = bool(os.environ.get("GOOGLE_APPLICATION_CREDENTIALS")
                    and os.path.exists(os.environ["GOOGLE_APPLICATION_CREDENTIALS"]))
VERTEX_MODEL = os.environ.get("DEMO_LLM_MODEL", "vertex_ai/gemini-2.5-flash")
FI_READY = all(os.environ.get(k) for k in ("FI_API_KEY", "FI_SECRET_KEY", "FI_BASE_URL"))
VAPI_READY = bool(os.environ.get("VAPI_API_KEY") and os.environ.get("VAPI_ASSISTANT_ID"))
RETELL_READY = bool(os.environ.get("RETELL_API_KEY") and os.environ.get("RETELL_AGENT_ID"))

print("env file loaded:", loaded)
print("vertex:", VERTEX_READY, " platform:", FI_READY,
      " vapi:", VAPI_READY, " retell:", RETELL_READY)

## 1. The registries — what's plugged in

The whole system is dispatched by three registries: name → factory. Adding a
provider, an environment, or an agent kind means **registering**, never editing
the engine. Nothing here is hardcoded — this cell just asks the registries what
they currently know.

In [ ]:
print("environments :", sorted(S.environment_registry.names()))

# Endpoint profiles are the ActorSources + voice targets. Each carries a manifest
# of capabilities and semantic flags (is this a code actor? a SIP target?).
for name in ["system_prompt", "factory", "http", "framework",
             "vapi_websocket", "retell_webcall", "webrtc"]:
    p = S.get_profile(name)
    if p is None:
        continue
    print(f"  {name:<16} turn_based={getattr(p, 'is_turn_based_target', '?')!s:<5} "
          f"runs_caller_code={getattr(p, 'runs_caller_code', '?')}")

## 2. An environment you can hold — a world with an action space

An **environment** owns the world and its action space. `EnvironmentAdapter` is
the contract: `reset()` publishes the tools + initial state, `handle_tool_call()`
executes an action and mutates state. Here `RefundWorld` exposes two tools
(`lookup_order`, `approve_refund`) and tracks a refund's status — a tiny
executable world, no credentials.

In [ ]:
from typing import Any, Mapping, Optional
from fi.simulate.environment import EnvironmentAdapter, EnvironmentSnapshot, ToolExecutionResult

TOOLS = [
    {"name": "lookup_order", "description": "Look up an order by id."},
    {"name": "approve_refund", "description": "Approve a refund for an order."},
]

class RefundWorld(EnvironmentAdapter):
    name = "refund_world"

    def __init__(self):
        self.state = {"refund": {"status": "pending"}}

    def reset(self, **_ctx) -> EnvironmentSnapshot:
        self.state = {"refund": {"status": "pending"}}
        return EnvironmentSnapshot(tools=list(TOOLS), state=dict(self.state))

    def handle_tool_call(self, tool_call: Mapping[str, Any], **_ctx) -> Optional[ToolExecutionResult]:
        name = tool_call.get("name") or (tool_call.get("function") or {}).get("name")
        cid = tool_call.get("id") or tool_call.get("tool_call_id")
        if name == "lookup_order":
            return ToolExecutionResult(tool_call_id=cid, tool_name=name,
                                       content="order A1: eligible for refund", result={"eligible": True})
        if name == "approve_refund":
            self.state["refund"]["status"] = "approved"
            return ToolExecutionResult(tool_call_id=cid, tool_name=name,
                                       content="refund approved", result={"status": "approved"},
                                       state_updates={"refund": {"status": "approved"}})
        return None

print("world tools:", [t["name"] for t in TOOLS])

### Drop a tool-calling agent into that world (offline, deterministic)

The agent is just an object with `async call(AgentInput) -> AgentResponse`. We
resolve it through the **`factory` ActorSource** — the same vocabulary a manifest
`agent:` block uses (`target` = `module:Class`) — so the harness constructs it
exactly the way a real hosted job would. Then the *one* `SimulationRunner` drives
it against `RefundWorld`. The agent calls `approve_refund`; the world executes it
and moves to `approved`.

In [ ]:
class ToolCallingRefundAgent:
    # Scripted target: look up the order, then approve the refund.
    async def call(self, agent_input: AgentInput) -> AgentResponse:
        turn = agent_input.turn_index
        if turn == 0:
            return AgentResponse(content="Let me look up your order.",
                                 tool_calls=[{"id": "c0", "name": "lookup_order",
                                              "arguments": {"order_id": "A1"}}])
        if turn == 1:
            return AgentResponse(content="It's eligible — approving the refund now.",
                                 tool_calls=[{"id": "c1", "name": "approve_refund",
                                              "arguments": {"order_id": "A1"}}])
        return AgentResponse(content="Your refund is approved. Anything else?")

# Expose the class as "module:attr" so the factory ActorSource resolves it the way
# a real job would (works in Jupyter and in plain execution).
import sys, types
_agents = sys.modules.setdefault("demo_v2_agents", types.ModuleType("demo_v2_agents"))
_agents.ToolCallingRefundAgent = ToolCallingRefundAgent

target = S.get_profile("factory").resolve_target(
    {"target": "demo_v2_agents:ToolCallingRefundAgent", "factory": True}, hosted=False)

spec = S.SimulationSpec(
    run_id="demo_refund_world",
    environment=S.EnvironmentSpec(adapter=S.EnvironmentAdapters.CHAT,
                                  world_kind=S.WorldKinds.CONVERSATION,
                                  config={"max_turns": 3, "min_turns": 1}),
    target=S.AgentEndpointSpec(adapter=S.TargetAdapters.FACTORY),
    simulator=S.SimulatorPolicySpec(adapter=S.SimulatorAdapters.SYNTHETIC_USER),
    scenario=S.Scenario(name="refund", dataset=[
        S.Persona(persona={"name": "Sam"}, situation="My order A1 arrived damaged.",
                  outcome="the refund is approved")]),
)

world = RefundWorld()
report = await S.SimulationRunner().run(spec, target=target, environment=world)  # Jupyter: top-level await
print("run status      :", report.status)
print("world final     :", world.state["refund"]["status"])
print("tool drove world:", world.state["refund"]["status"] == "approved")
print("\n" + report.test_cases[0].result.transcript)

## 3. Drop in **any** agent — the ActorSource surface

`RefundWorld` used a Python class via `factory`. Every other way you'd hand us an
agent is *also* an ActorSource resolved through the same registry. You never edit
the engine — you declare what you have:

```python
S.get_profile("system_prompt").resolve_target({"system_prompt": "...", "model": "gpt-4o"})
S.get_profile("factory").resolve_target({"target": "mypkg.agents:Support", "factory": True})
S.get_profile("import_object").resolve_target({"target": "mypkg.agents:instance"})
S.get_profile("http").resolve_target({"url": "https://my-agent/turn"})
S.get_profile("framework").resolve_target({"target": "mypkg:graph"})  # LangGraph/CrewAI/…
```

…or skip the registry entirely and pass any object with `.call()` straight to the
runner (next section). Same `target=` slot either way.

> **Security — code actors don't run on prod in-process.** ActorSource kinds that
> load *your* code (`factory`, `import_object`, `framework`, callable) are
> **deny-by-default in hosted runs** (`profile.runs_caller_code == True`). Locally
> (`hosted=False`) they run in-process for your convenience; hosted, they are
> rejected until they go through the code-executor sandbox container. `http` and
> `system_prompt` are the safe hosted kinds (no caller code in-process).

In [ ]:
# Which ActorSource kinds are safe to run untrusted in a hosted run?
for name in ["system_prompt", "http", "factory", "import_object", "framework"]:
    p = S.get_profile(name)
    if p:
        code_actor = getattr(p, "runs_caller_code", None)
        print(f"  {name:<14} hosted_safe={'no' if code_actor else 'yes'}")

## 4. A proper chat simulation against a **real LLM**

Now a real back-and-forth. The target is a live LLM (Vertex Gemini via litellm),
dropped in as a plain object — the plug-and-play story end to end. The `chat`
environment drives a synthetic user against it and records the transcript. Needs
Vertex creds; skips cleanly otherwise.

In [ ]:
class LiteLLMAgent:
    # Any object with async .call() is a valid target - here a litellm LLM.
    def __init__(self, model, system_prompt):
        self.model, self.system_prompt = model, system_prompt

    async def call(self, agent_input: AgentInput) -> AgentResponse:
        import litellm
        messages = [{"role": "system", "content": self.system_prompt}]
        for m in agent_input.messages:
            role = "assistant" if m["role"] in ("assistant", "agent") else "user"
            messages.append({"role": role, "content": m["content"]})
        resp = await litellm.acompletion(model=self.model, messages=messages,
                                         temperature=0.3, max_tokens=800)  # thinking model: headroom
        return AgentResponse(content=resp["choices"][0]["message"]["content"])


chat_report = None
if VERTEX_READY:
    llm_target = LiteLLMAgent(VERTEX_MODEL,
        "You are a concise, friendly delivery-support agent. Acknowledge the issue, "
        "give a status, and offer a clear next step.")
    chat_spec = S.SimulationSpec(
        run_id="demo_chat_llm",
        environment=S.EnvironmentSpec(adapter=S.EnvironmentAdapters.CHAT,
                                      world_kind=S.WorldKinds.CONVERSATION,
                                      config={"max_turns": 4, "min_turns": 2, "modality": "text"}),
        target=S.AgentEndpointSpec(adapter=S.TargetAdapters.CALLABLE),
        simulator=S.SimulatorPolicySpec(adapter=S.SimulatorAdapters.SYNTHETIC_USER),
        scenario=S.Scenario(name="late-delivery", dataset=[
            S.Persona(persona={"name": "Morgan", "role": "customer"},
                      situation="A delivery is 3 days late; ask for status and ETA.",
                      outcome="Get a clear status and a concrete next step.")]),
    )
    chat_report = await S.SimulationRunner().run(chat_spec, target=llm_target)
    print("status:", chat_report.status)
    print(chat_report.test_cases[0].result.transcript[:1400])
else:
    print("Vertex not configured — skipping the live chat sim.")

## 5. Register your **own** environment

Environments are plugins too. `@register_environment("name")` adds a world to the
registry; the *same* `SimulationRunner` then drives it — no engine changes. (For
distribution, a package advertises it under the `fi.simulate.environments`
entry-point group and it auto-discovers on install.) Here a trivial echo world,
registered and run through the identical spine.

In [ ]:
from fi.simulate.environments.base import EnvironmentManifest
from fi.simulate.runtime.capabilities import EndpointCapabilities
from fi.simulate.simulation.models import TestReport, TestCaseResult

@S.register_environment("echo_world")
class EchoWorldPlugin:
    manifest = EnvironmentManifest(name="echo_world", world_kinds=["conversation"],
                                   capabilities=EndpointCapabilities(text=True))

    async def run(self, spec, *, target=None, **_):
        persona = spec.scenario.dataset[0]
        line = f"echo: {persona.situation}"
        return TestReport(results=[TestCaseResult(
            persona=persona, transcript=f"User: {persona.situation}\nAgent: {line}",
            messages=[{"role": "user", "content": persona.situation},
                      {"role": "assistant", "content": line}])])

print("registered:", "echo_world" in S.environment_registry.names())
echo_spec = S.SimulationSpec(
    run_id="demo_echo",
    environment=S.EnvironmentSpec(adapter="echo_world",  # custom registered name -> raw string
                                  world_kind=S.WorldKinds.CONVERSATION, config={}),
    target=S.AgentEndpointSpec(adapter=S.TargetAdapters.CALLABLE),
    simulator=S.SimulatorPolicySpec(adapter=S.SimulatorAdapters.SYNTHETIC_USER),
    scenario=S.Scenario(name="echo", dataset=[
        S.Persona(persona={"name": "Dev"}, situation="hello from a custom world", outcome="echoed")]),
)
echo_report = await S.SimulationRunner().run(echo_spec)
print(echo_report.test_cases[0].result.transcript)

## 6. Eval the run

Every environment produces the **same** report shape, so one evaluator grades them
all. `evaluate_agent_report` scores the trajectory on ~38 metrics offline (no LLM
call); metrics whose requirement isn't configured are excluded from the aggregate.

In [ ]:
report_to_grade = chat_report if chat_report is not None else report
# evaluate_agent_report grades the legacy trajectory shape; SimulationReport.to_legacy()
# projects the unified report back to it.
ev = S.evaluate_agent_report(report_to_grade.to_legacy(), threshold=0.7)
c0 = ev.cases[0]
applicable = [m for m in c0.metrics if m.applicable]
print(f"aggregate score: {ev.score}   passed: {ev.passed}")
print(f"applicable: {len(applicable)}   excluded (n/a): {len(c0.metrics) - len(applicable)}")
for m in sorted(applicable, key=lambda x: x.score)[:8]:
    print(f"  {m.score:>6}  {m.name}")

### Platform evals as assertions (`fi_eval`)

The metrics above are local. A `fi_eval` assertion instead scores output with a
**hosted FutureAGI eval template** — the same evals the platform runs — dispatched
via `FI_API_KEY` / `FI_SECRET_KEY` / `FI_BASE_URL`. Needs the platform reachable.

In [ ]:
from fi.alk import evals

if VERTEX_READY and FI_READY:
    suite = {
        "version": "agent-learning.eval.v1", "name": "fi-eval-demo",
        "providers": [{"id": "vertex", "type": "vertex",
                       "model": VERTEX_MODEL.split("/")[-1],
                       "vertex_project": os.environ.get("GOOGLE_CLOUD_PROJECT"),
                       "vertex_location": os.environ.get("GOOGLE_CLOUD_LOCATION", "us-central1"),
                       "temperature": 0, "max_tokens": 2000}],  # thinking model: leave headroom
        "prompts": [{"id": "p", "template":
            'Return ONLY a raw JSON object (no markdown, no code fences) with keys '
            '"ticket" (string) and "summary" (string, <= 12 words) for: {{ticket}}'}],
        "tests": [{"id": "invoice", "vars": {"ticket": "customer charged twice on one invoice"},
                   "assert": [{"type": "fi_eval", "eval": "is_json", "threshold": 0.5}]}],
    }
    res = evals.run_eval_suite(suite, suite_path=".")
    print("status:", res["status"])
    for c in res["evaluation"]["cases"]:
        print("  output:", c["output"][:80])
        for a in c["assertions"]:
            if a.get("type") == "fi_eval":
                print(f"  fi_eval {a['eval']}: score={a['score']} passed={a['passed']}")
else:
    print("Set Vertex + FI_API_KEY/FI_SECRET_KEY/FI_BASE_URL to run platform evals.")

## 7. Voice — the **same** runner, a real provider call

The headline of the refactor: voice is no longer a separate engine you call by
hand. A voice run is a `SimulationSpec` with `environment.adapter="voice"`; the
target provider is chosen by the **target adapter string** (`vapi_websocket`,
`retell_webcall`, `webrtc`, …) — a registered endpoint profile, not a hardcoded
branch. The FutureAGI simulator (persona voice) runs on LiveKit with Vertex + a
speech provider, and the identical `SimulationRunner` drives the call.

`build_voice_spec(provider)` below assembles that spec the same way the hosted
runner does (typed voice inputs ride secret-free in `environment.config`;
providers referenced by `*_env` name). **Opt-in** — a real, billable call:
set `RUN_VOICE_DEMO=1`.

In [ ]:
RUN_VOICE_DEMO = os.environ.get("RUN_VOICE_DEMO") == "1"

def _simulator_cfg():
    return {
        "llm": {"provider": os.environ.get("SIMULATOR_LLM_PROVIDER", "google"),
                "model": os.environ.get("SIMULATOR_LLM_MODEL", "gemini-2.5-flash-lite")},
        "stt": {"provider": os.environ.get("SIMULATOR_STT_PROVIDER", "deepgram"),
                "model": os.environ.get("SIMULATOR_STT_MODEL", "nova-2"), "language": "en"},
        "tts": {"provider": os.environ.get("SIMULATOR_TTS_PROVIDER", "deepgram"),
                "model": os.environ.get("SIMULATOR_TTS_MODEL", "aura-asteria-en"),
                "voice": os.environ.get("SIMULATOR_TTS_VOICE", "aura-asteria-en")},
    }

def _agent_def(provider, run_id):
    if provider == "vapi":
        return {"name": "vapi-web-target",
                "system_prompt": os.environ.get("VAPI_TARGET_SYSTEM_PROMPT", "You are a support agent."),
                "target": {"provider": "vapi", "assistant_id": os.environ.get("VAPI_ASSISTANT_ID"),
                           "api_key_env": "VAPI_API_KEY"},
                "transport": {"kind": "vapi_websocket"},
                "provider_evidence": {"provider": "vapi", "call_id_source": "originator_response"}}
    if provider == "retell":
        return {"name": "retell-web-target",
                "system_prompt": os.environ.get("RETELL_TARGET_SYSTEM_PROMPT", "You are a support agent."),
                "target": {"provider": "retell", "agent_id": os.environ.get("RETELL_AGENT_ID"),
                           "api_key_env": "RETELL_API_KEY"},
                "transport": {"kind": "retell_webcall"},
                "provider_evidence": {"provider": "retell", "call_id_source": "originator_response"}}
    raise ValueError(provider)

def build_voice_spec(provider):
    # ExecutionPolicy must clear the voice call's own budget — same computation the
    # hosted runner's _build_voice_spec uses.
    from fi.simulate.runtime.spec import ExecutionPolicy, TimeoutPolicy
    from fi.simulate.runtime import new_run_id

    run_id = new_run_id()
    agent_def = _agent_def(provider, run_id)
    kind = agent_def["transport"]["kind"]
    scenario = S.Scenario(name=f"{provider}-late-delivery", dataset=[
        S.Persona(persona={"name": "Morgan", "role": "customer"},
                  situation="A delivery is late. Ask for its status, ETA, and the next action.",
                  outcome="Complete a natural multi-turn conversation and close politely.")])
    runtime = {"url": os.environ["LIVEKIT_URL"], "room_name": f"demo-{provider}-{run_id}",
               "room_mode": "managed"}
    params = {"record_audio": True, "min_turn_messages": 6, "max_seconds": 150,
              "conversation_direction": "simulator_first"}
    run_seconds = max(300.0, params["max_seconds"] + 15 + 30 + 30 + 60)
    return S.SimulationSpec(
        run_id=run_id,
        environment=S.EnvironmentSpec(adapter=S.EnvironmentAdapters.VOICE,
                                      world_kind=S.WorldKinds.VOICE_TELEPHONY, config={
            "agent_definition": agent_def, "livekit_runtime": runtime,
            "simulator": _simulator_cfg(), "params": params}),
        target=S.AgentEndpointSpec(adapter=kind),  # runtime transport string ("vapi_websocket"/"retell_webcall") stays plain
        simulator=S.SimulatorPolicySpec(adapter=S.SimulatorAdapters.LIVEKIT_SIMULATOR),
        scenario=scenario,
        execution=ExecutionPolicy(timeout=TimeoutPolicy(run_seconds=run_seconds)),
    )

print("build_voice_spec ready. RUN_VOICE_DEMO =", RUN_VOICE_DEMO)

### 7a. Vapi web call

In [ ]:
vapi_report = None
if RUN_VOICE_DEMO and VAPI_READY and os.environ.get("LIVEKIT_URL"):
    vapi_report = await S.SimulationRunner().run(build_voice_spec("vapi"))
    print("vapi status:", vapi_report.status)
    tc = vapi_report.test_cases[0]
    print("case:", tc.status)
    print(tc.result.transcript[:1200])
else:
    print("Skipped. Set RUN_VOICE_DEMO=1 and Vapi creds to place the call.")

### 7b. Retell web call

In [ ]:
retell_report = None
if RUN_VOICE_DEMO and RETELL_READY and os.environ.get("LIVEKIT_URL"):
    retell_report = await S.SimulationRunner().run(build_voice_spec("retell"))
    print("retell status:", retell_report.status)
    tc = retell_report.test_cases[0]
    print("case:", tc.status)
    print(tc.result.transcript[:1200])
else:
    print("Skipped. Set RUN_VOICE_DEMO=1 and Retell creds to place the call.")

## 8. Submit to the platform (hosted ingestion)

Pass a `FutureAGIResultSink` as `result_sink=` and the runner POSTs the finished
run to the ALK ingestion API, where the platform recomputes conversation metrics +
CSAT + cost and renders it next to native runs. `FI_RUN_TEST_ID` selects the
target run-test; `FI_TEST_EXECUTION_ID` (optional) submits into a pre-created
execution — the exact path the hosted **SimulationRunnerWorkflow** uses when the
platform triggers the SDK itself.

```python
from fi.simulate.results import FutureAGIResultSink
sink = FutureAGIResultSink(root=".fagi/runs")           # reads FI_* from env
report = await S.SimulationRunner().run(build_voice_spec("vapi"), result_sink=sink)
# → writes locally AND submits; check .fagi/runs/<run_id>/submission.json
```

This is the **same** sink + ingestion contract for chat and voice — the report
shape is identical, so the platform doesn't care which environment produced it.

## 9. Simulation is simulation — not just chat/voice

Chat and voice are **interaction modalities**, not the definition of a simulation. The
core is modality-free: an **environment** owns a world + an **action space** + a notion
of success. A Text2SQL benchmark is a peer of chat, not a special case — the agent's
action space is `run_sql` / `inspect_schema`, its **observation** is the returned rows
or the SQL error, the world **state** is `solved: true`, and scoring is the world
contract (did the final state hit the goal), not CSAT. Same `SimulationSpec`, different
world; `spec.environment.config` carries `{db_uri, schema, gold_rows}` instead of
`{livekit_runtime, simulator}`.

```python
class Text2SQLWorld(EnvironmentAdapter):
    def reset(self, **_):
        return EnvironmentSnapshot(tools=[{"name": "inspect_schema"}, {"name": "run_sql"}],
                                   state={"schema": self.schema, "question": self.q})
    def handle_tool_call(self, call, **_):
        if call["name"] == "run_sql":
            rows = self.db.execute(call["arguments"]["query"])   # real execution = observation
            solved = rows == self.gold                            # correctness = world state
            return ToolExecutionResult(tool_name="run_sql", content=str(rows),
                                       state_updates={"solved": solved, "rows": rows})
```

`RefundWorld` in §2 already proved the non-conversational tool-world path end to end
(scored on state, not talk); **`examples/sdk_text2sql_world.py`** is the full runnable
version — a real in-memory SQLite world dropped in as an `EnvironmentAdapter`, driven by
the ordinary `chat` loop.

**Tool mocking is a *world capability*, not an environment of its own.** Any loop can
carry it: pass a live world object (above), **or** declare mocked tools right in
`spec.environment.config` — JSON-serializable, so it survives a hosted job. Any target
that calls `approve_refund` gets the mocked result; no live object needed:

```python
environment=S.EnvironmentSpec(
    adapter=S.EnvironmentAdapters.CHAT, world_kind=S.WorldKinds.CONVERSATION,
    config={"mock_tools": {"approve_refund": {"content": "refund approved",
              "state_updates": {"refund": {"status": "approved"}}}}})
```

**On magic strings:** every adapter slot accepts a plain string *or* the matching
enum — `S.EnvironmentAdapters`, `S.TargetAdapters`, `S.SimulatorAdapters`,
`S.WorldKinds`. The enum member **is** the string (same `spec_hash`), so you get
autocomplete + typo-safety for the built-ins while custom registered names (like
`"echo_world"`) stay plain strings. This notebook uses the enums throughout.

## Glossary — every term used here

**The 5 primitives (the spine)**

| Term | What it is |
|---|---|
| **Environment** | the *world* the agent acts in — owns the action space + what "good" means (`chat`, `voice`, a Text2SQL world). Not the agent, not the test. |
| **Agent / Target** | the thing under test — your bot. "Target" = target of the simulation. Any shape: prompt, class, HTTP endpoint, LangGraph. |
| **Actor** | the agent, framed as an actor that *acts in* an environment. |
| **Scenario** | the *episode setup* — a list of situations to test; holds one or more Personas. |
| **Persona** | one test case — *who* the synthetic user is + *situation* + desired *outcome*. |

**Who drives it**

| Term | What it is |
|---|---|
| **Simulator / synthetic user** | the *fake customer* the kit generates to poke your agent (persona-driven). The opponent; your agent is under test. |

**Contract + engine**

| Term | What it is |
|---|---|
| **SimulationSpec** | the *frozen recipe* for one run: environment + target + simulator + scenario. Declarative, secret-free. |
| **SimulationRunner** | the *engine*. `runner.run(spec)` → executes the episode → returns a report. One runner for every environment. |
| **Registry** | a *phone book*: name → factory (`"voice"` → the voice plugin). Add a provider = new entry, not an engine edit. Three: environments / endpoints / simulators. |

**The environment's two contracts**

| Term | What it is |
|---|---|
| **EnvironmentPlugin** | code that *owns the episode loop* and returns a report (`chat`/`voice` are plugins). |
| **EnvironmentAdapter** | the *action-space + state* contract of a world: `reset()` (publish tools + state), `observe()` (current view), `handle_tool_call()` (run one action, mutate world). |
| **Action space** | the moves the agent can make = the **tools** the world exposes (`run_sql`, `approve_refund`). |
| **Observation** | what the agent *sees* after acting (rows, an error, state) — feeds its next move. |
| **State** | the world's ground truth (`refund.status = approved`). Scoring reads this. |
| **EnvironmentSnapshot** | the data `reset`/`observe` return: `{tools, state}`. |
| **ToolExecutionResult** | what `handle_tool_call` returns: `{content, result, state_updates}`. |

**Plugging your agent in**

| Term | What it is |
|---|---|
| **ActorSource** | the *adapter kind* that turns "the thing you have" into a target: `system_prompt`, `factory` (your class), `import_object` (a live object), `http`, `framework`. |
| **EndpointProfile** | the *record* behind a target name: capabilities + flags (`is_sip`, `runs_caller_code`) + how to build it. `get_profile("vapi_websocket")`. |
| **Transport / adapter** | the *channel* to the target: `vapi_websocket`, `retell_webcall`, `webrtc`, `http`, `callable`. |
| **wrap_agent** | helper: take any object with `.call()`, make it a valid target. |

**Scoring**

| Term | What it is |
|---|---|
| **World-contract / goal_machine** | scores against the *world's own success test* (did state hit the goal), not "was the chat nice." |
| **settle** | score *at episode end* (final state) vs per-step. Voice is settle-only. |
| **evaluate_agent_report** | offline scorer, ~38 trajectory metrics (task completion, tool use, safety). No LLM call. |
| **CSAT** | customer-satisfaction score the platform computes per conversation (`overall_score`). |
| **fi_eval** | an assertion that scores output with a *hosted FutureAGI eval template* (`is_json`, `toxicity`). |

**Platform / hosted**

| Term | What it is |
|---|---|
| **ResultSink** | the *pipe* that POSTs a finished report to the platform to render + recompute metrics/CSAT. |
| **RunTest** | a *saved simulation definition* on the platform (agent + scenario). Runs create **TestExecution**s; each conversation = a **CallExecution**. |
| **Hosted runner** | the path where the *platform triggers the SDK* as a job instead of you running it locally. |
| **StartRunnerJob** | the *job envelope* the hosted runner consumes (spec + sink config + secret references). |
| **child_entrypoint** | the process the runner spawns to run the SDK and submit. |

**Security**

| Term | What it is |
|---|---|
| **runs_caller_code** | profile flag: does this target run *your* code in-process? `factory`/`import_object`/`framework` = yes → **deny-by-default in hosted** (local or sandbox only). `http`/`system_prompt` = no → hosted-safe. |

**CLI**

| Term | What it is |
|---|---|
| **Manifest** | a *JSON file* describing environment + agent + scenario — the CLI's front door (`agent-learn simulate run manifest.json`). Same concepts as `SimulationSpec`, built into a spec underneath. |

## Where to go next

| Task | New API | Facade helper (v1) |
|------|-------------|--------------------|
| Chat sim | `SimulationRunner().run(spec, target=obj)` | `run_local_text_manifest` |
| Voice sim | `SimulationRunner().run(voice_spec)` | `run_voice_simulation` |
| Drop in an agent | `get_profile(kind).resolve_target(cfg)` | `wrap_agent(obj)` |
| New environment | `@register_environment("name")` | — |
| Eval | `evaluate_agent_report(report)` | `agent-learn eval-artifact` |
| Submit | `result_sink=FutureAGIResultSink(...)` | same |

- **One spine**: chat and voice both flow through `SimulationRunner` + `SimulationSpec`.
- **Nothing hardcoded**: providers, environments, and agent kinds are all registry entries.
- **Secret-free specs**: configs reference secrets by `*_env` name; the runner resolves them.